In [1]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import hnsw_benchmark as bm
import farhest_point_order as fp
from datasets import load_dataset
import os
from time import time

d:\Dokumentumok\MSc\F3_TudLab\EX9BLV_HNSW_Preordering\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
# Alapadatok megadása
# dataset_name = "arguana"
dataset_name = "scidocs"

# embedding_model_group = "BAAI"
# embedding_model = "bge-base-en"
# embedding_model = "bge-small-en"                      # megvan LID
# embedding_model = "bge-base-en-v1.5"
# embedding_model = "bge-small-en-v1.5"

# embedding_model_group = "infgrad"
# embedding_model = "stella-base-en-v2"                 # megvan LID

# embedding_model_group = "intfloat"
# embedding_model = "e5-base"
# embedding_model = "e5-small"
# embedding_model = "e5-base-v2"
# embedding_model = "e5-small-v2"                       # megvan LID Farthest LPCA TwoNN
# embedding_model = "multilingual-e5-large"             # megvan LID Farthest
# embedding_model = "multilingual-e5-base"
# embedding_model = "multilingual-e5-small"

# embedding_model_group = "llmrails"
# embedding_model = "ember-v1"

embedding_model_group = "sentence-transformers"
embedding_model = "all-MiniLM-L6-v2"                  # megvan LID

# embedding_model_group = "TaylorAI"
# embedding_model = "bge-micro"

# embedding_model_group = "thenlper"
# embedding_model = "gte-base"

calc_lid = True
calc_fps = False

In [27]:
# Adathalmazok beolvasása
# Corpus
corpus = load_dataset("mteb/" + dataset_name, "corpus")
corpus_texts = corpus.data["corpus"]["text"]
print(f"Corpus size: {len(corpus_texts):10d}")

# Query
queries = load_dataset("mteb/" + dataset_name, "queries")
query_texts = queries.data["queries"]["text"]
print(f"Query size:  {len(query_texts):10d}")

Corpus size:      25657
Query size:        1000


In [28]:
print(corpus_texts[0])
print(query_texts[0])

An evolutionary recurrent network which automates the design of recurrent neural/fuzzy networks using a new evolutionary learning algorithm is proposed in this paper. This new evolutionary learning algorithm is based on a hybrid of genetic algorithm (GA) and particle swarm optimization (PSO), and is thus called HGAPSO. In HGAPSO, individuals in a new generation are created, not only by crossover and mutation operation as in GA, but also by PSO. The concept of elite strategy is adopted in HGAPSO, where the upper-half of the best-performing individuals in a population are regarded as elites. However, instead of being reproduced directly to the next generation, these elites are first enhanced. The group constituted by the elites is regarded as a swarm, and each elite corresponds to a particle within it. In this regard, the elites are enhanced by PSO, an operation which mimics the maturing phenomenon in nature. These enhanced elites constitute half of the population in the new generation, 

In [29]:
# Embedding végrehajtása
model = SentenceTransformer(embedding_model_group + "/" + embedding_model)

corpus_embeddings = model.encode(
    [f"passage: {text}" for text in corpus_texts],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

query_embeddings = model.encode(
    [f"query: {text}" for text in query_texts],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)


corpus_embeddings = np.asarray(corpus_embeddings, dtype=np.float32)
query_embeddings = np.asarray(query_embeddings, dtype=np.float32)

Batches: 100%|██████████| 32/32 [00:04<00:00,  7.08it/s]


In [30]:
print(f"Corpus dimensions: {corpus_embeddings.shape}")
print(f"Query dimensions:  {query_embeddings.shape}")

Corpus dimensions: (25657, 384)
Query dimensions:  (1000, 384)


In [31]:
# LID számolás
if calc_lid:
    corpus_lids = bm.calculate_lid(corpus_embeddings)

d:\Dokumentumok\MSc\F3_TudLab\EX9BLV_HNSW_Preordering\code\hnsw_benchmark.py:140: RuntimeWarning: divide by zero encountered in scalar power
  lid_values[i] = -((sum_log / k) ** -1)


In [32]:
# Farthest points igazi számítása
if calc_fps:
    corpus_farthest_scores = fp.farthest_point_order(corpus_embeddings)

In [33]:
# Fájlok kiírása
# Corpus
df_corpus = pd.DataFrame(columns=["vectors"], data=[[v] for v in corpus_embeddings])
corpus_filename = dataset_name + "_corpus_" + embedding_model

if calc_lid:
    df_corpus = df_corpus.assign(LID=corpus_lids)
    corpus_filename = corpus_filename + "_lid"
if calc_fps:
    df_corpus = df_corpus.assign(fp_score=corpus_farthest_scores)
    corpus_filename = corpus_filename + "_fps"

corpus_output_path = os.path.join("..\\data\\", corpus_filename + ".json")
df_corpus.to_json(corpus_output_path, orient="records", lines=True)


# Query
df_query = pd.DataFrame(columns=["vectors"], data=[[v] for v in query_embeddings])
query_filename = dataset_name + "_query_" + embedding_model
query_output_path = os.path.join("..\\data\\", query_filename + ".json")
df_query.to_json(query_output_path, orient="records", lines=True)

In [35]:
print(df_corpus.keys())
print(len(corpus_lids))

Index(['vectors', 'LID'], dtype='str')
25657


In [38]:
print(np.mean(corpus_lids))

-inf


In [44]:
# KNN számítás minden pontra
k = 100
start = time()
data = np.asarray([np.asarray(v) for v in corpus_embeddings], dtype=np.float32)
print(f"Init:         {((time() - start) * 1000):.2f} ms")
start = time()
scores = data @ data.T
print(f"Inversion:    {((time() - start) * 1000):.2f} ms")

# Assign the same ID to identical vectors, then exclude them from each row.
start = time()
_, vector_ids = np.unique(data, axis=0, return_inverse=True)
for row, vector_id in enumerate(vector_ids):
    scores[row, vector_ids == vector_id] = -np.inf
print(f"Filter 0:     {((time() - start) * 1000):.2f} ms")

start = time()
topk = np.argpartition(
    -scores,
    kth=k-1,
    axis=1,
)[:, :k]
print(f"TopK:         {((time() - start) * 1000):.2f} ms")

# Optional: sort those k results by actual score
start = time()
rows = np.arange(len(data))[:, None]
order = np.argsort(-scores[rows, topk], axis=1)

knn_matrix = np.take_along_axis(topk, order, axis=1)
print(f"Sort:         {((time() - start) * 1000):.2f} ms")

# LID számítása egyes vektorokra
lid_values = np.full(len(data), np.nan, dtype=np.float64)

sumdist = 0
sumcalc = 0
start = time()
for i, neighbor_indices in enumerate(knn_matrix):
    #Távolságok számítása
    sd = time()
    distances = np.array([bm.euclidian_distance(data[i], data[j]) for j in neighbor_indices])

    distances = distances[distances > 0]

    if len(distances) < k:
        print(f"Point: {i}")
        print(knn_matrix[i][:10])
        print(neighbor_indices[:10])
        raise ValueError(f"Some elements deleted.")
    sumdist += time() - sd

    # LID érték kiszámítása
    sc = time()
    r_k = distances[-1]
    sum_log = np.sum(np.log2(distances / r_k)) # ln lenne helyes, de mindegy a rangsorolás szempontjából, mivel mindegyik log fv. szig. mon. növ.
    if sum_log == 0 or k == 0:
        print(i)
        print(neighbor_indices[:10])
        exit
    lid_values[i] = -((sum_log / k) ** -1)
    sumcalc = time() - sc

print(f"LID all:      {((time() - start) * 1000):.2f} ms")
print(f"  LID calc:   {(sumcalc * 1000):.2f} ms")
print(f"  dist calc:  {(sumdist * 1000):.2f} ms")

Init:         216.78 ms


MemoryError: Unable to allocate 2.45 GiB for an array with shape (25657, 25657) and data type float32

In [ ]:
i = 12
neighbor_indices = knn_matrix[i]
distances = np.array([bm.euclidian_distance(data[i], data[j]) for j in neighbor_indices])